In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Load bashrc environment for cached models
import subprocess
result = subprocess.run(['bash', '-c', 'source ~/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Verify HuggingFace cache directories
print("HF_HOME:", os.environ.get('HF_HOME', 'Not set'))
print("TRANSFORMERS_CACHE:", os.environ.get('TRANSFORMERS_CACHE', 'Not set'))

# Check GPU availability
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

HF_HOME: /net/projects2/chai-lab/shared_models
TRANSFORMERS_CACHE: Not set



CUDA available: True
GPU device: NVIDIA A40


# Code Evaluation for Gendered Pronoun Resolution Circuit Analysis

## Repository: `/net/scratch2/smallyan/pronoun_claude_2025-12-26_01-36-15`

This notebook evaluates all code blocks from the main analysis notebook `2025-12-26-01-36_GenderedPronounCircuit.ipynb`.

The notebook contains:
- **47 total cells**
- **37 code cells** 
- **10 markdown cells**

Code cells to evaluate: Cells 1, 2, 4, 6, 7, 8, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 44, 46

## Evaluation Criteria

For each code block:
1. **Runnable (Y/N)**: Block executes without error
2. **Correct-Implementation (Y/N)**: Logic matches stated purpose
3. **Redundant (Y/N)**: Block duplicates another computation
4. **Irrelevant (Y/N)**: Block doesn't contribute to project goal

## Cell 1: Setup and imports

In [3]:
# Cell 1: Setup and imports
import os
os.chdir('/net/scratch2/smallyan/pronoun_claude_2025-12-26_01-36-15')

import torch
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A40
Memory: 47.7 GB


**Cell 1 Result**: Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N

## Cell 2: Load GPT-2 small via HookedTransformer

In [4]:
# Cell 2: Load GPT-2 small via HookedTransformer
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Model loaded: {model.cfg.model_name}")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads per layer: {model.cfg.n_heads}")
print(f"d_model: {model.cfg.d_model}")
print(f"d_head: {model.cfg.d_head}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Model loaded: gpt2
Layers: 12
Heads per layer: 12
d_model: 768
d_head: 64


**Cell 2 Result**: Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N

## Cell 4: Create logs directory and plan file

In [5]:
# Cell 4: Create logs directory and plan file
os.makedirs('logs', exist_ok=True)

plan_content = """# Gendered Pronoun Resolution Circuit Analysis Plan

## Goal
Identify the circuit in GPT-2 small that performs gendered pronoun resolution - matching pronouns like "he" and "she" to their correct antecedents based on grammatical gender agreement.

## Hypothesis
The gendered pronoun resolution circuit consists of three main components:

1. **Gender Detection Heads**: Early-mid layer attention heads and MLPs that identify and encode gender information from gendered nouns (explicit gender like "man/woman" or implicit via stereotypes like "nurse/engineer").

2. **Pronoun Matching Heads**: Mid-layer attention heads active at pronoun positions that attend back to gendered entities and use the encoded gender information to select the matching antecedent.

3. **Entity Copying Heads**: Later-layer attention heads similar to IOI Name-Mover heads that copy information about the selected antecedent to predict subsequent tokens.

We also hypothesize that:
- The model will show stereotypical gender bias (stronger performance on stereotype-congruent cases)
- Some heads may overlap with the IOI circuit, but gender-specific heads will be distinct
- Gender information will be encoded differently than positional information in IOI

## Method

### 1. Dataset Creation
- Create 100+ sentence pairs testing pronoun resolution
- Balance: 50+ stereotypical, 50+ counter-stereotypical
- Include minimal pairs differing only in pronoun
- Include multi-entity disambiguation cases

### 2. Baseline Evaluation
- Measure model's pronoun resolution accuracy
- Compare stereotypical vs counter-stereotypical performance
- Quantify bias metric

### 3. Activation Patching
- Use mean activation patching to identify important heads
- Patch from clean to corrupted runs (pronoun swap)
- Identify heads where patching most affects performance

### 4. Attention Pattern Analysis
- Visualize attention from pronoun positions
- Identify heads attending to gendered antecedents
- Compare stereotypical vs counter-stereotypical attention

### 5. Gender Probing
- Train linear probes on activations to decode gender
- Identify layers where gender information appears
- Find gender-encoding dimensions

### 6. Circuit Extraction
- Combine evidence to identify circuit components
- Categorize heads by role
- Respect write budget constraint

## Expected Outputs
- real_circuits_1.json with circuit nodes
- Visualizations of circuit components
- Quantified bias metrics
"""

with open('logs/plan.md', 'w') as f:
    f.write(plan_content)

print("Created logs/plan.md")
print(f"File size: {os.path.getsize('logs/plan.md')} bytes")

Created logs/plan.md
File size: 2446 bytes


**Cell 4 Result**: Runnable=Y, Correct=Y, Redundant=N, Irrelevant=N

## Cell 6: Define stereotypical gender associations for occupations